In [17]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("DecisionTreeExample")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [22]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('D:\my-repo\data\students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

In [23]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# Кодиране на категорийни колони
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex", handleInvalid="keep")
degree_indexer = StringIndexer(inputCol="Degree", outputCol="DegreeIndex", handleInvalid="keep")
platform_indexer = StringIndexer(inputCol="PlatformUsage", outputCol="PlatformUsageIndex", handleInvalid="keep")
format_indexer = StringIndexer(inputCol="PreferredFormat", outputCol="PreferredFormatIndex", handleInvalid="keep")
# Кодиране на целевия клас
label_indexer = StringIndexer(inputCol="Satisfaction", outputCol="label", handleInvalid="keep")
# Обединяване на признаците в една колона features
assembler = VectorAssembler(
    inputCols=["GenderIndex","DegreeIndex","PlatformUsageIndex","PreferredFormatIndex","Age", "PlatformHours","OutPlatformHours","Self-assessment"],
    outputCol="features")

In [24]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Decision tree модел
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features",
                            maxDepth=5, impurity="gini")

# Pipeline
pipeline = Pipeline(stages=[
    gender_indexer,
    degree_indexer,
    platform_indexer,
    format_indexer,
    label_indexer,
    assembler,
    dt
])

# Разделяне на данните
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

# Обучение
model = pipeline.fit(train_df)

# Прогнозиране
predictions = model.transform(test_df)

# точност
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
    metricName="accuracy" )
# F1
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
    metricName="f1")
# Precision (претеглена)
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
    metricName="weightedPrecision")
# Recall (претеглена)
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
    metricName="weightedRecall")

accuracy = evaluator_acc.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)
precision = evaluator_prec.evaluate(predictions)
recall = evaluator_rec.evaluate(predictions)
print("Accuracy =", accuracy)
print("Precision =", precision)
print("Recall =", recall)
print("F1-score =", f1)

Accuracy = 0.5
Precision = 0.3333333333333333
Recall = 0.5
F1-score = 0.375


In [152]:
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Decision tree модел за регресия или RandomForest 
dtr = DecisionTreeRegressor(labelCol="Satisfaction", featuresCol="features", 
                            maxDepth=5, impurity="variance")

# Pipeline
pipeline = Pipeline(stages=[
    gender_indexer,
    degree_indexer,
    platform_indexer,
    format_indexer,
    assembler,
    dtr
])

# Разделяне на данните
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

# Обучение
model = pipeline.fit(train_df)

# Прогнозиране
predictions = model.transform(test_df)

# Оценка
evaluator_rmse = RegressionEvaluator(
    labelCol="Satisfaction",
    predictionCol="prediction",
    metricName="rmse" )
evaluator_r2 = RegressionEvaluator(
    labelCol="Satisfaction",
    predictionCol="prediction",
    metricName="r2" )

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("RMSE =", rmse)
print("R² =", r2)

RMSE = 0.6755206325494435
R² = 0.6349374999999998


In [12]:
from pyspark.ml.classification import RandomForestClassifier

# Random Forest модел
rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                            numTrees=10, maxDepth=5, impurity="gini"
)

# Pipeline
pipeline = Pipeline(stages=[
    gender_indexer,
    degree_indexer,
    platform_indexer,
    format_indexer,
    label_indexer,
    assembler,
    rf
])

# Разделяне на данните
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

# Обучение
model = pipeline.fit(train_df)

# Прогнозиране
predictions = model.transform(test_df)

# Оценка
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy" )

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")

accuracy = evaluator_acc.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print("Accuracy =", accuracy)
print("F1-score =", f1)

Accuracy = 0.5
F1-score = 0.375


In [27]:
from pyspark.sql.functions import col, when
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# превръщаме probability във временен масив
pred_with_arr = predictions.withColumn(
    "prob_arr",
    vector_to_array(col("probability"))
)

classes = [row["label"] for row in predictions.select("label").distinct().orderBy("label").collect()]

auc_values = []
auprc_values = []

for i, c in enumerate(classes):
    # текущият клас е положителен, останалите - отрицателни
    pred_binary = pred_with_arr.withColumn(
        "binaryLabel",
        when(col("label") == c, 1.0).otherwise(0.0)
    ).withColumn(
        "score",
        col("prob_arr")[i]
    )

    binary_evaluator = BinaryClassificationEvaluator(
        labelCol="binaryLabel",
        rawPredictionCol="score"
    )

    auc = binary_evaluator.evaluate(
        pred_binary,
        {binary_evaluator.metricName: "areaUnderROC"}
    )

    auprc = binary_evaluator.evaluate(
        pred_binary,
        {binary_evaluator.metricName: "areaUnderPR"}
    )

    auc_values.append(auc)
    auprc_values.append(auprc)

auc_macro = sum(auc_values) / len(auc_values)
auprc_macro = sum(auprc_values) / len(auprc_values)

print("ROC AUC (macro) =", auc_macro)
print("AUPRC (macro) =", auprc_macro)

ROC AUC (macro) = 0.6666666666666667
AUPRC (macro) = 0.4583333333333333


In [14]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

spark = SparkSession.builder.appName("ClassificationMetrics").getOrCreate()

data = [
    (30.0, 70.0, 1.0),
    (32.0, 65.0, 1.0),
    (28.0, 80.0, 0.0),
    (35.0, 60.0, 1.0),
    (36.0, 58.0, 1.0),
    (31.0, 72.0, 0.0),
    (29.0, 85.0, 0.0),
    (40.0, 55.0, 1.0),
    (38.0, 57.0, 1.0),
    (27.0, 90.0, 0.0)
]

df = spark.createDataFrame(data, ["Temperature", "Humidity", "label"])

assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity"],
    outputCol="features"
)

df_features = assembler.transform(df)

train_data, test_data = df_features.randomSplit([0.7, 0.3], seed=42)

model = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=3
).fit(train_data)

predictions = model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

predictionAndLabels = predictions.select("prediction", "label") \
    .rdd.map(lambda row: (float(row["prediction"]), float(row["label"])))

metrics = MulticlassMetrics(predictionAndLabels)
confusion = metrics.confusionMatrix().toArray()

print("Confusion matrix:\n", confusion)
print("Accuracy =", accuracy)
print("Weighted Precision =", precision)
print("Weighted Recall =", recall)
print("F1 =", f1)

Confusion matrix:
 [[1. 0.]
 [0. 2.]]
Accuracy = 1.0
Weighted Precision = 1.0
Weighted Recall = 1.0
F1 = 1.0


In [15]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

binary_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction")

roc_auc = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderROC"})
auprc = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderPR"})

print("ROC AUC =", roc_auc)
print("AUPRC =", auprc)

ROC AUC = 1.0
AUPRC = 1.0
